# Explore Bronze

---

## Setup e Imports

In [1]:
import sys
sys.path.insert(0, "../src")

from pathlib import Path
from nlst_datalake.config import (
    PATHOLOGY_ROOT, PATHOLOGY_SOURCES,
    LUNG_CANCER_ROOT, LUNG_CANCER_SOURCES,
)

import openslide
import pydicom
import polars as pl

---

## Inventário da camada bronze

In [2]:
# Contar arquivos por categoria
print("=" * 60)
print("INVENTÁRIO DA BRONZE")
print("=" * 60)

print("\n── PATOLOGIA (.SVS) ──")
total_svs = 0
for label, path in PATHOLOGY_SOURCES.items():
    count = len(list(path.rglob("*.svs")))
    total_svs += count
    print(f"  {label}: {count} arquivos")
print(f"  TOTAL: {total_svs} arquivos .SVS")

print("\n── LUNG CANCER (DICOM) ──")
total_patients = 0
total_dicoms = 0
for label, path in LUNG_CANCER_SOURCES.items():
    # Pacientes = pastas no nível batch/sub_batch/PID
    patients = set()
    dicoms = 0
    for f in path.rglob("*"):
        if f.is_file():
            dicoms += 1
            # PID é a pasta 3 níveis acima do arquivo (batch/sub_batch/PID/T0/series/file)
            # Ou podemos pegar pelo nome da pasta que é numérica
            parts = f.relative_to(path).parts
            if len(parts) >= 3:
                patients.add(parts[2])  # batch/sub_batch/PID
    total_patients += len(patients)
    total_dicoms += dicoms
    print(f"  {label}: {len(patients)} pacientes, {dicoms} arquivos")
print(f"  TOTAL: {total_patients} pacientes, {total_dicoms} arquivos DICOM")

INVENTÁRIO DA BRONZE

── PATOLOGIA (.SVS) ──
  adenocarcinoma: 553 arquivos
  squamous_cell: 326 arquivos
  bronchio_alveolar: 155 arquivos
  other: 191 arquivos
  TOTAL: 1225 arquivos .SVS

── LUNG CANCER (DICOM) ──
  screen_detected: 623 pacientes, 353811 arquivos
  other: 422 pacientes, 313132 arquivos
  TOTAL: 1045 pacientes, 666943 arquivos DICOM


---

## Explorar um arquivo .SVS

In [3]:
# Pegar o primeiro .SVS disponível
sample_svs = next(PATHOLOGY_ROOT.rglob("*.svs"))
print(f"Arquivo: {sample_svs.name}")
print(f"Tamanho: {sample_svs.stat().st_size / (1024**2):.1f} MB")
print(f"Subtipo: {sample_svs.relative_to(PATHOLOGY_ROOT).parts[0]}")
print()

slide = openslide.OpenSlide(str(sample_svs))

print("── Propriedades do OpenSlide ──")
for key, value in sorted(slide.properties.items()):
    print(f"  {key}: {value}")

print(f"\n── Dimensões ──")
print(f"  Dimensões (pixels): {slide.dimensions}")
print(f"  Níveis: {slide.level_count}")
print(f"  Dimensões por nível: {slide.level_dimensions}")
print(f"  Downsamples: {slide.level_downsamples}")

slide.close()

Arquivo: NLSI0000238.svs
Tamanho: 221.1 MB
Subtipo: 2-4-Other-Lung-Cancers

── Propriedades do OpenSlide ──
  aperio.AppMag: 20
  aperio.DSR ID: PDL-Mylo
  aperio.Date: 05/26/09
  aperio.Filename: NLSI0000238
  aperio.Focus Offset: -0.000500
  aperio.ICC Profile: ScanScope v1
  aperio.ImageID: 15806
  aperio.Left: 27.886496
  aperio.LineAreaXOffset: 0.000000
  aperio.LineAreaYOffset: 0.000000
  aperio.LineCameraSkew: 0.000153
  aperio.MPP: 0.5021
  aperio.OriginalWidth: 42840
  aperio.Originalheight: 35991
  aperio.ScanScope ID: PDL-APERIO
  aperio.StripeWidth: 2040
  aperio.Time: 14:38:51
  aperio.Title: LSSLung
  aperio.Top: 21.881935
  aperio.User: ca4989ad-6015-4355-b0c8-600bb482f0ca
  openslide.associated.macro.height: 481
  openslide.associated.macro.width: 1280
  openslide.associated.thumbnail.height: 768
  openslide.associated.thumbnail.icc-size: 141992
  openslide.associated.thumbnail.width: 895
  openslide.comment: Aperio Image Library v10.0.44
42840x35991 [0,0 41867x35891] (

---

## Definir campos de metadados para Patologia (Silver)

In [4]:
# Baseado na exploração acima, definir quais campos extrair
# Ajuste esta lista depois de ver o output da célula 3!

PATHOLOGY_FIELDS = [
    # Identificação
    "slide_id",           # nome do arquivo (sem extensão)
    "cancer_subtype",     # pasta pai (adenocarcinoma, squamous_cell, etc.)
    "batch",              # pasta do batch
    "source_file",        # path completo do .SVS

    # Dimensões da imagem
    "width",              # slide.dimensions[0]
    "height",             # slide.dimensions[1]
    "level_count",        # slide.level_count

    # Propriedades do OpenSlide (as mais relevantes — ajustar após exploração)
    "vendor",             # openslide.vendor
    "mpp_x",             # openslide.mpp-x (microns per pixel)
    "mpp_y",             # openslide.mpp-y
    "objective_power",    # openslide.objective-power (magnificação)
    "comment",            # openslide.comment (pode conter metadata adicional)

    # Metadados do arquivo
    "file_size_mb",       # tamanho do arquivo
]

print("Campos planejados para Silver (pathology_metadata):")
for f in PATHOLOGY_FIELDS:
    print(f"  - {f}")

Campos planejados para Silver (pathology_metadata):
  - slide_id
  - cancer_subtype
  - batch
  - source_file
  - width
  - height
  - level_count
  - vendor
  - mpp_x
  - mpp_y
  - objective_power
  - comment
  - file_size_mb


---

## Explorar um arquivo DICOM (metadados pydicom)

In [5]:
# Encontrar o primeiro arquivo DICOM
# DICOMs do NLST geralmente não têm extensão .dcm — são nomeados pelo SOP Instance UID
sample_dicom_dir = next(LUNG_CANCER_ROOT.rglob("T0"))  # Pegar um timepoint T0
dicom_files = [f for f in sample_dicom_dir.rglob("*") if f.is_file()]

if not dicom_files:
    # Tentar um nível mais fundo (T0/series_uid/arquivo)
    sample_patient = next(iter(next(iter(LUNG_CANCER_SOURCES.values())).iterdir())).iterdir()
    print("Ajuste necessário — verificar estrutura manualmente")
else:
    sample_dcm = dicom_files[0]
    print(f"Arquivo: {sample_dcm.name[:60]}...")
    print(f"Tamanho: {sample_dcm.stat().st_size / 1024:.1f} KB")
    print()

    ds = pydicom.dcmread(str(sample_dcm), stop_before_pixels=True)

    print("── Metadados DICOM ──")
    for elem in ds:
        if elem.VR != "SQ":  # Pular sequências (muito verbosas)
            print(f"  ({elem.tag}) {elem.keyword}: {elem.value}")

Arquivo: 1.2.840.113654.2.55.53905640941549003240914952204335660855.d...
Tamanho: 1464.3 KB

── Metadados DICOM ──
  ((0008,0005)) SpecificCharacterSet: ISO_IR 100
  ((0008,0008)) ImageType: ['ORIGINAL', 'PRIMARY', 'LOCALIZER']
  ((0008,0016)) SOPClassUID: 1.2.840.10008.5.1.4.1.1.2
  ((0008,0018)) SOPInstanceUID: 1.2.840.113654.2.55.53905640941549003240914952204335660855
  ((0008,0020)) StudyDate: 19990102
  ((0008,0030)) StudyTime: 
  ((0008,0050)) AccessionNumber: 954223
  ((0008,0060)) Modality: CT
  ((0008,0070)) Manufacturer: GE MEDICAL SYSTEMS
  ((0008,0090)) ReferringPhysicianName: 
  ((0008,1010)) StationName: G09
  ((0008,1030)) StudyDescription: NLST-LSS
  ((0008,103E)) SeriesDescription: 0,OPL,GE,HSQX,EXPERIMENTAL7,null,na,80,0,0,na
  ((0008,1090)) ManufacturerModelName: HiSpeed QX/i
  ((0010,0010)) PatientName: 073403^LSS
  ((0010,0020)) PatientID: 120352
  ((0010,0030)) PatientBirthDate: 
  ((0010,0040)) PatientSex: 
  ((0010,1000)) OtherPatientIDs: 073403
  ((0012,0010)) 

---

## Definir campos de metadados para Lung Cancer (Silver)

In [6]:
# Baseado na exploração acima, definir quais campos extrair
# Ajuste esta lista depois de ver o output da célula 5!

LUNG_CANCER_FIELDS = [
    # Identificação
    "pid",                    # Patient ID (pasta do paciente)
    "timepoint",              # T0, T1, T2 (pasta do timepoint)
    "series_uid",             # Series Instance UID (pasta da série)
    "sop_instance_uid",       # SOP Instance UID (nome do arquivo ou tag DICOM)
    "cancer_category",        # screen_detected ou other

    # Informações do paciente (do DICOM header)
    "patient_id_dicom",       # (0010,0020) Patient ID
    "patient_sex",            # (0010,0040) Patient's Sex
    "patient_age",            # (0010,1010) Patient's Age

    # Informações do estudo
    "study_date",             # (0008,0020) Study Date
    "study_description",      # (0008,1030) Study Description
    "modality",               # (0008,0060) Modality (CT, etc.)

    # Informações da série
    "series_description",     # (0008,103E) Series Description
    "series_number",          # (0020,0011) Series Number

    # Parâmetros de aquisição
    "slice_thickness",        # (0018,0050) Slice Thickness
    "pixel_spacing",          # (0028,0030) Pixel Spacing
    "kvp",                    # (0018,0060) KVP
    "rows",                   # (0028,0010) Rows
    "columns",                # (0028,0011) Columns
    "reconstruction_diameter",# (0018,1100) Reconstruction Diameter

    # Metadados do arquivo
    "source_file",            # path completo
    "file_size_kb",           # tamanho do arquivo
]

print("Campos planejados para Silver (lung_cancer_metadata):")
for f in LUNG_CANCER_FIELDS:
    print(f"  - {f}")

Campos planejados para Silver (lung_cancer_metadata):
  - pid
  - timepoint
  - series_uid
  - sop_instance_uid
  - cancer_category
  - patient_id_dicom
  - patient_sex
  - patient_age
  - study_date
  - study_description
  - modality
  - series_description
  - series_number
  - slice_thickness
  - pixel_spacing
  - kvp
  - rows
  - columns
  - reconstruction_diameter
  - source_file
  - file_size_kb


---

## Testar extração em batch (preview)

In [ ]:
# Testar extração de 5 SVS para validar antes de rodar em tudo
from nlst_datalake.extraction import extract_svs_metadata, extract_dicom_metadata

# Patologia — 5 amostras
svs_files = list(PATHOLOGY_ROOT.rglob("*.svs"))[:5]
df_svs_sample = extract_svs_metadata(svs_files)
print("── Sample Pathology Metadata ──")
print(df_svs_sample)

print()

# Lung Cancer — 5 amostras
dicom_files_sample = list(LUNG_CANCER_ROOT.rglob("*"))
dicom_files_sample = [f for f in dicom_files_sample if f.is_file() and not f.name.startswith(".")][:5]
df_dcm_sample = extract_dicom_metadata(dicom_files_sample)
print("── Sample Lung Cancer Metadata ──")
